In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm
import os

np.random.seed(42)

# Step 1: Load Initial Data
def load_data(function_number, base_path):
    inputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_inputs.npy')
    outputs_path = os.path.join(base_path, f'function_{function_number}', 'initial_outputs.npy')
    
    inputs = np.load(inputs_path)
    outputs = np.load(outputs_path)

    print(function_number)
    #print(inputs)
    #print(outputs)

    """
    if function_number == 1:
       wk_query_point = np.array([[0.883890, 0.983890],[0.374540, 0.950713]])
       wk_output = np.array([9.590330531222263e-135],[-1.5622772406409273e-117])
    elif function_number == 2:
       wk_query_point = np.array([[0.926564, 1.026564],[0.747504, 0.208970]])
       wk_output = np.array([-0.041995540546406196],[0.2803303123560871])
    elif function_number == 3:
       wk_query_point = np.array([[1.065994, 1.041359, 1.090881],[0.403482, 0.382170, 0.489363]])
       wk_output = np.array([-0.769427956661122],[-0.03310307977430594])
    elif function_number == 4:
       wk_query_point = np.array([[1.085621, 1.019592, 1.039177, 1.099482],[0.000001, 0.000001, 0.124558, 0.000001]])
       wk_output = np.array([-67.60493430274798],[-22.782193418373407])
    elif function_number == 5:
       wk_query_point = np.array([[0.936477, 0.962540, 0.979484, 1.057643],[0.999999, 0.999999, 0.000001, 0.000001]])
       wk_output = np.array([7713.373609304799],[1616.6257474282386])
    elif function_number == 6:
       wk_query_point = np.array([[1.057739, 1.031871, 1.078805, 1.061655, 0.992819]],[0.183405, 0.304243, 0.524756, 0.431945, 0.291230])
       wk_output = np.array([-2.868905011263093],[-0.9489046340640067])
    elif function_number == 7:
       wk_query_point = np.array([[1.042450, 1.024693, 1.024570, 1.061017, 1.098654, 1.051013]
                                 ,[0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371]])
       wk_output = np.array([0.000004636858051500375],[1.680828424430851])
    elif function_number == 8:
       wk_query_point = np.array([[1.085945, 1.073979, 1.098885, 1.002985, 1.086901, 1.090243, 1.092914, 1.092915]
                                 ,[0.000001, 0.205513, 0.000001, 0.084257, 0.808305, 0.355768, 0.000001, 0.452510]])
       wk_output = np.array([1.6498596481450019],[9.8188854584285])

    inputs = np.vstack([inputs_initial, wk_query_point])
    outputs = np.concatenate([outputs_initial, wk_output])
    
    return inputs, outputs
    """

# Step 2: Define the Gaussian Process Model
def define_gp_model():
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42, normalize_y=True)
    return gp

# Step 3: Fit the Model
def fit_model(gp, inputs, outputs):
    gp.fit(inputs, outputs)
    return gp

# Step 4a: Upper Confidence Bound (UCB)
def acquisition_ucb(x, model, kappa=2.0):
    x = np.atleast_2d(x)
    mean, std = model.predict(x, return_std=True)
    return mean + kappa * std   # We will MAXIMIZE this

# Step 4b: Expected Improvement (EI)
def acquisition_ei(x, model, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mean, std = model.predict(x, return_std=True)
    std = std.reshape(-1, 1)
    
    with np.errstate(divide='warn'):
        imp = mean - y_best - xi
        Z = imp / std
        ei = imp * norm.cdf(Z) + std * norm.pdf(Z)
        ei[std == 0.0] = 0.0
    return ei.ravel()

# Step 5: Optimize Acquisition Function (maximize it)
def suggest_next_point(bounds, model, acq_type='EI', y_best=None):
    # Define objective (negative for minimization)
    def objective(x):
        if acq_type == 'UCB':
            return -acquisition_ucb(x, model)
        elif acq_type == 'EI':
            return -acquisition_ei(x, model, y_best)
        else:
            raise ValueError("Invalid acquisition type. Choose 'UCB' or 'EI'.")
    
    result = minimize(objective,
                      x0=np.random.uniform(bounds[:, 0], bounds[:, 1]),
                      bounds=bounds,
                      method='L-BFGS-B')
    return result.x

# Step 6: Format Output
def format_output(point):
    return '-'.join([f"{x:.6f}" for x in point])

# Step 7: Main Pipeline
def run_pipeline(function_number, base_path, acq_type):
    inputs, outputs = load_data(function_number, base_path)
    gp = define_gp_model()
    gp = fit_model(gp, inputs, outputs)
    
    bounds = np.array([[0.000001, 0.999999]] * inputs.shape[1])  # Assuming normalized input range [0,1]
    y_best = np.max(outputs)
    
    next_point = suggest_next_point(bounds, gp, acq_type=acq_type, y_best=y_best)
    formatted_output = format_output(next_point)
    
    print(f"Function {function_number} [{acq_type}] → Suggested next point: {formatted_output}")
    print(f"Inputs producing current best: {inputs[np.argmax(outputs)]}")
    print(f"Current best y value: {y_best}\n")

# Step 8: Run for all functions
base_path = "initial_data"

# You can switch between 'UCB' and 'EI' here
for i in range(1, 9):
    run_pipeline(i, base_path, acq_type='UCB')   # or acq_type='UCB'


1


TypeError: cannot unpack non-iterable NoneType object